# Courbe d'inflation à 50 ans — Modèle Nelson-Siegel

Ce notebook calibre un modèle de **Nelson-Siegel (NS)** pour construire une courbe d'inflation annuelle sur 50 ans, à partir de 4 inputs utilisateur :

1. **Input 1** : taux d'inflation initial (année 0)
2. **Input 2** : taux d'inflation à long terme (année 50) — la courbe doit y converger
3. **Input 3** : (année, taux cible) — point de calibration intermédiaire n°1
4. **Input 4** : (année, taux cible) — point de calibration intermédiaire n°2

### Modèle

$$y(t) = b_0 + b_1 \cdot \frac{1-e^{-t/\alpha}}{t/\alpha} + b_2 \left(\frac{1-e^{-t/\alpha}}{t/\alpha} - e^{-t/\alpha}\right)$$

- $b_0$ = niveau long terme (input 2)
- $b_1$ = input 1 − input 2 (garantit $y(0)$ = input 1)
- $b_2, \alpha$ = calibrés par optimisation (moindres carrés) sur les points input 3, input 4, et input 2 à l'année 50 (pour forcer la convergence effective à horizon fini).

## 1. Installation / imports

In [ ]:
# Sur Colab, scipy et matplotlib sont déjà installés par défaut.
# Cette ligne ne fait rien si c'est déjà le cas, sinon elle installe ce qu'il faut.
!pip install -q scipy matplotlib numpy pandas

In [ ]:
import numpy as np
from scipy.optimize import least_squares
import matplotlib.pyplot as plt
import pandas as pd

## 2. Fonction Nelson-Siegel

In [ ]:
def nelson_siegel(t, b0, b1, b2, alpha):
    """
    Calcule le taux NS pour une ou plusieurs maturités t (en années).
    Gère proprement la limite en t -> 0.
    """
    t = np.asarray(t, dtype=float)
    x = t / alpha

    # f1(t) = (1 - exp(-x)) / x, avec limite f1(0) = 1
    f1 = np.where(x == 0, 1.0, (1 - np.exp(-np.where(x == 0, 1.0, x))) / np.where(x == 0, 1.0, x))
    f2 = f1 - np.exp(-x)

    return b0 + b1 * f1 + b2 * f2

## 3. Calibration de b₂ et α par moindres carrés

In [ ]:
def calibrate_ns(b0, b1, target_points, x0=(0.10, 2.0),
                  bounds=([-1.0, 1.0], [1.0, 50.0])):
    """
    target_points : liste de tuples (maturité_en_années, taux_cible)
                    ex : [(3, 0.03), (21, 0.021), (50, 0.02)]
    x0            : valeurs initiales fictives (b2, alpha)

    NB sur les bornes (bounds) : alpha est contraint à >= 1.0. Sans cette
    contrainte, l'optimiseur peut trouver un alpha très petit (< 0.5) qui
    ajuste un peu mieux les 3 points cibles mais produit une courbe avec
    une légère "bosse" non monotone entre t=0 et le premier point cible
    (la composante de courbure b2 devient alors très locale). La borne
    alpha >= 1.0 donne une courbe économiquement plus réaliste (décroissance
    /convergence lisse) pour un coût quasi nul en qualité d'ajustement
    (écarts < 0.1 pt de % sur les points cibles). Élargissez les bornes si
    vous voulez explicitement autoriser plus de courbure à court terme.
    Retourne (b2_opt, alpha_opt, résultat_optimisation)
    """
    t_arr = np.array([p[0] for p in target_points], dtype=float)
    y_arr = np.array([p[1] for p in target_points], dtype=float)

    def residuals(params):
        b2, alpha = params
        model_vals = nelson_siegel(t_arr, b0, b1, b2, alpha)
        return model_vals - y_arr

    result = least_squares(residuals, x0=list(x0), bounds=bounds)
    b2_opt, alpha_opt = result.x
    return b2_opt, alpha_opt, result

## 4. Construction complète de la courbe à partir des 4 inputs

In [ ]:
def build_inflation_curve(infl_0, infl_50,
                           annee_a, infl_a,
                           annee_b, infl_b,
                           horizon=50,
                           b2_alpha_init=(0.10, 2.0)):
    """
    infl_0   : input 1 - taux d'inflation initial (t=0), ex. 0.04
    infl_50  : input 2 - taux d'inflation long terme (t=50), ex. 0.02
    annee_a, infl_a : input 3 - (année, taux cible), ex. (3, 0.03)
    annee_b, infl_b : input 4 - (année, taux cible), ex. (21, 0.021)
    horizon  : nombre d'années de la courbe (défaut 50)
    """
    # Paramètres fixés par construction
    b0 = infl_50
    b1 = infl_0 - infl_50

    # Points de calibration pour b2 et alpha :
    # les deux points définis par l'utilisateur + le niveau long terme
    # (pour forcer la convergence effective à l'horizon 50 ans)
    target_points = [(annee_a, infl_a), (annee_b, infl_b), (horizon, infl_50)]

    b2, alpha, opt_res = calibrate_ns(b0, b1, target_points, x0=b2_alpha_init)

    # Courbe annuelle sur l'horizon demandé
    years = np.arange(0, horizon + 1)
    curve = nelson_siegel(years, b0, b1, b2, alpha)

    params = {"b0": b0, "b1": b1, "b2": b2, "alpha": alpha}
    return years, curve, params, opt_res, target_points

## 5. Inputs utilisateur

👉 Modifiez les 4 valeurs ci-dessous selon votre scénario.

In [ ]:
# ---- INPUTS UTILISATEUR (à modifier) ----
infl_0 = 0.04                 # input 1 : inflation initiale (t=0)
infl_50 = 0.02                # input 2 : inflation long terme (t=50)
annee_a, infl_a = 3, 0.03     # input 3 : (année, taux cible)
annee_b, infl_b = 21, 0.021   # input 4 : (année, taux cible)
# ------------------------------------------

## 6. Calcul de la courbe et vérification

In [ ]:
years, curve, params, opt_res, target_points = build_inflation_curve(
    infl_0, infl_50, annee_a, infl_a, annee_b, infl_b
)

print("Paramètres calibrés du modèle Nelson-Siegel :")
for k, v in params.items():
    print(f"  {k:6s} = {v:.6f}")
print(f"\nConvergence optimisation : {opt_res.success} ({opt_res.message})")

print("\nVérification aux points cibles :")
for t, target in target_points:
    modeled = nelson_siegel(t, **params)
    print(f"  Année {t:>3.0f} : cible = {target:6.3%}  |  modèle = {modeled:6.3%}  "
          f"|  écart = {(modeled - target)*100:+.3f} pts")

In [ ]:
df = pd.DataFrame({"Année": years, "Inflation": curve})
df["Inflation (%)"] = (df["Inflation"] * 100).round(3)
df[["Année", "Inflation (%)"]]

## 7. Graphique

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(years, curve * 100, label="Courbe d'inflation (Nelson-Siegel)", color="#2563eb", linewidth=2)
plt.scatter([0], [infl_0 * 100], color="#dc2626", zorder=5, label=f"Input 1 (t=0) : {infl_0:.1%}")
plt.scatter([annee_a], [infl_a * 100], color="#16a34a", zorder=5, label=f"Input 3 (t={annee_a}) : {infl_a:.1%}")
plt.scatter([annee_b], [infl_b * 100], color="#ea580c", zorder=5, label=f"Input 4 (t={annee_b}) : {infl_b:.1%}")
plt.scatter([50], [infl_50 * 100], color="#7c3aed", zorder=5, label=f"Input 2 (t=50) : {infl_50:.1%}")
plt.axhline(infl_50 * 100, color="gray", linestyle="--", linewidth=1, alpha=0.6)

plt.title("Courbe d'inflation annuelle sur 50 ans — Modèle Nelson-Siegel")
plt.xlabel("Année")
plt.ylabel("Taux d'inflation (%)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. (Optionnel) Export de la courbe en CSV

Pratique pour réutiliser les résultats ailleurs (Excel, autre modèle, etc.).

In [ ]:
df[["Année", "Inflation (%)"]].to_csv("courbe_inflation_nelson_siegel.csv", index=False)
print("Fichier exporté : courbe_inflation_nelson_siegel.csv")

# Sur Colab, pour télécharger directement le fichier :
# from google.colab import files
# files.download("courbe_inflation_nelson_siegel.csv")